# Cheatsheet — Klasifikasi Multi-Class (>2 kelas)

**Kapan pakai file ini:** lebih dari dua kelas, tapi tiap dokumen tetap punya **satu** label. Contoh: topik berita (olahraga/ekonomi/politik/teknologi), rating 1–5, kategori keluhan.

Alur tetap, tinggal ubah **§1 CONFIG**: struktur file, kolom, preprocessing, model.
Feature extraction dikunci di **TF-IDF** (default paling aman untuk klasifikasi teks).

```
intip file -> ambil kolom -> preprocessing -> TF-IDF -> model -> evaluasi -> output
```

Teori tiap langkah: `cheatsheet_klasifikasi_teks.ipynb` · Semua opsi: `latihan_sklearn.ipynb`

---
## §1 · CONFIG — ubah di sini saja

In [1]:
# ---------------- DATA: struktur file ----------------
PATH   = "data/berita_topik.tsv"
SEP    = None        # file .tsv -> otomatis tab. Isi "	" atau ";" kalau perlu paksa
HEADER = "infer"
ENC    = None

# ---------------- DATA: kolom mana yang dipakai ----------------
TEXT_COL   = None    # None = deteksi otomatis; di file ini "judul"
LABEL_COL  = None    # None = deteksi otomatis; di file ini "kategori"
ID_COL     = None
LABEL_MAP  = None    # kalau kelas berupa angka: {0: "olahraga", 1: "ekonomi", ...}
MULTILABEL = False

BAHASA = "id"        # "en" | "id"

# ---------------- PREPROCESSING (True/False) ----------------
LOWERCASE   = True
MASK        = True
STOPWORD    = True
JAGA_NEGASI = True
STEMMING    = False     # untuk bahasa Indonesia butuh: pip install Sastrawi

# ---------------- FEATURE EXTRACTION: TF-IDF ----------------
NGRAM  = (1, 2)
MIN_DF = 1              # data kecil -> jangan buang term langka

# ---------------- MODEL ----------------
MODEL       = "svm"     # nb | logreg | svm | tree | rf
SEIMBANGKAN = False

TEST_SIZE, SEED = 0.3, 42

---
## §2 · Intip struktur file dulu

Jalankan ini **sebelum** apa pun. Kalau nama kolom, separator, atau encoding-nya tidak seperti
dugaan, perbaiki di CONFIG lalu jalankan ulang sel ini. Jangan menebak struktur file.

In [2]:
import re, numpy as np, pandas as pd

# ---- 1. lihat 3 baris pertama file MENTAH (sebelum pandas menafsirkannya) ----
print("=== isi mentah 3 baris pertama ===")
with open(PATH, encoding="utf-8", errors="replace") as f:
    for i, baris in zip(range(3), f):
        print(f"  {i}: {baris.rstrip()[:110]}")

# ---- 2. baca dengan setelan di CONFIG ----
_sep = SEP or ("\t" if PATH.endswith((".tsv", ".tab")) else ",")
for _enc in ([ENC] if ENC else ["utf-8", "latin-1", "cp1252"]):
    try:
        raw = pd.read_csv(PATH, sep=_sep, header=HEADER, encoding=_enc,
                          engine="python", on_bad_lines="skip")
        break
    except (UnicodeDecodeError, UnicodeError):
        continue

print(f"\n=== terbaca: {raw.shape[0]} baris x {raw.shape[1]} kolom "
      f"(sep={_sep!r}, encoding={_enc}) ===")
print("nama kolom :", list(raw.columns))
print("\njumlah nilai unik per kolom (kolom label biasanya yang paling sedikit):")
print(raw.nunique().to_string())
print("\n=== 3 baris pertama ===")
print(raw.head(3).to_string())

# ---- 3. saran isian CONFIG -- periksa dulu, kalau benar tinggal disalin ----
_teks = [c for c in raw.columns if raw[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
_tebak_teks = max(_teks, key=lambda c: raw[c].astype(str).str.len().mean()) if _teks else None
_kand = [(raw[c].nunique(), c) for c in raw.columns
         if c != _tebak_teks and 2 <= raw[c].nunique() <= 200]
_tebak_label = min(_kand)[1] if _kand else None

print("\n=== saran untuk CONFIG (salin kalau tebakannya benar) ===")
print(f"TEXT_COL   = {_tebak_teks!r}")
print(f"LABEL_COL  = {_tebak_label!r}")
if _tebak_label is not None:
    _nilai = sorted(map(str, raw[_tebak_label].dropna().unique()))
    print(f"# {len(_nilai)} nilai unik di kolom label: {_nilai[:8]}")

=== isi mentah 3 baris pertama ===
  0: kategori	judul
  1: teknologi	Ponsel lipat generasi kedua hadir dengan engsel lebih tahan lama
  2: teknologi	Layanan komputasi awan perusahaan itu mengalami gangguan selama tiga jam

=== terbaca: 88 baris x 2 kolom (sep='\t', encoding=utf-8) ===
nama kolom : ['kategori', 'judul']

jumlah nilai unik per kolom (kolom label biasanya yang paling sedikit):
kategori     4
judul       88

=== 3 baris pertama ===
    kategori                                                                     judul
0  teknologi          Ponsel lipat generasi kedua hadir dengan engsel lebih tahan lama
1  teknologi  Layanan komputasi awan perusahaan itu mengalami gangguan selama tiga jam
2  teknologi         Mobil listrik otonom mulai diuji coba di jalan raya beberapa kota

=== saran untuk CONFIG (salin kalau tebakannya benar) ===
TEXT_COL   = 'judul'
LABEL_COL  = 'kategori'
# 4 nilai unik di kolom label: ['ekonomi', 'olahraga', 'politik', 'teknologi']


---
## §3 · Ambil kolom teks & label

`TEXT_COL` / `LABEL_COL` diisi `None` = dideteksi otomatis (kolom teks = string dengan rata-rata
terpanjang, kolom label = kolom lain dengan nilai unik paling sedikit). Kalau tebakannya salah,
tulis sendiri di CONFIG — nama kolom (`"message"`) atau indeks angka (`5`) sama-sama bisa.

In [3]:
# ---- tentukan kolom; kalau CONFIG diisi None, dideteksi otomatis ----
text_col, label_col = TEXT_COL, LABEL_COL

if text_col is None:      # kolom teks = kolom berisi string dengan rata-rata terpanjang
    kand = [c for c in raw.columns if raw[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
    if not kand:
        raise ValueError("kolom teks tidak terdeteksi -> isi TEXT_COL di CONFIG")
    text_col = max(kand, key=lambda c: raw[c].astype(str).str.len().mean())

if label_col is None:     # kolom label = kolom lain dengan nilai unik paling sedikit
    batas = 200 if MULTILABEL else 20
    kand = [(raw[c].nunique(), c) for c in raw.columns
            if c != text_col and 2 <= raw[c].nunique() <= batas]
    if not kand:
        raise ValueError("kolom label tidak terdeteksi -> isi LABEL_COL di CONFIG")
    label_col = min(kand)[1]

print(f"kolom teks  : {text_col!r}")
print(f"kolom label : {label_col!r}")
print(f"kolom id    : {ID_COL!r}" + ("  (tidak dipakai)" if ID_COL is None else ""))

# ---- rapikan jadi DataFrame dua/tiga kolom ----
petakan = {text_col: "text", label_col: "label"}
if ID_COL is not None:
    petakan[ID_COL] = "id"
df = raw.rename(columns=petakan)[list(petakan.values())].copy()

df["text"] = df["text"].astype(str).str.strip()
if not MULTILABEL:
    df["label"] = df["label"].apply(lambda v: v.strip().lower() if isinstance(v, str) else v)
if LABEL_MAP:
    df["label"] = df["label"].map(LABEL_MAP).fillna(df["label"])

n0 = len(df)
df = df[df["text"].str.len() >= 3]
df = df[~df["text"].str.lower().isin({"nan", "none", "na", "-"})]
df = df.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"\nsetelah dibersihkan: {n0} -> {len(df)} baris")

kolom teks  : 'judul'
kolom label : 'kategori'
kolom id    : None  (tidak dipakai)

setelah dibersihkan: 88 -> 88 baris


In [4]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

print("jumlah kelas:", df["label"].nunique())
print("distribusi  :", df["label"].value_counts().to_dict())
df.head(3)

jumlah kelas: 4
distribusi  : {'teknologi': 22, 'politik': 22, 'ekonomi': 22, 'olahraga': 22}


,text,label
0,Ponsel lipat generasi kedua hadir dengan engse...,teknologi
1,Layanan komputasi awan perusahaan itu mengalam...,teknologi
2,Mobil listrik otonom mulai diuji coba di jalan...,teknologi


---
## §4 · Preprocessing

Semua sakelar di CONFIG dibaca di sini. Kalau semuanya `False`, teks diteruskan apa adanya —
`TfidfVectorizer` tetap melakukan lowercase dan tokenisasi sendiri, jadi pipeline tetap jalan.

In [5]:
STOP_EN = {"i","me","my","we","you","your","he","she","it","they","them","this","that","is","are",
           "was","were","be","been","have","has","had","do","does","did","a","an","the","and","but",
           "if","or","because","as","of","at","by","for","with","to","from","in","out","on","off",
           "then","so","than","too","very","just","now","s","t","can","will","there","here","what",
           "when","how","all","any","am","been","its","our","their"}
STOP_ID = {"yang","dan","di","ke","dari","ini","itu","untuk","dengan","pada","adalah","ada","saya",
           "kamu","dia","kami","kita","mereka","akan","sudah","telah","juga","atau","karena","agar",
           "saja","oleh","sebagai","dalam","para","nya","banget","sekali","sangat","tetapi","tapi"}
NEGASI  = {"no","not","never","nor","cannot"} | {"tidak","bukan","tanpa","jangan","belum","kurang"}

stop = (STOP_EN if BAHASA == "en" else STOP_ID)
if JAGA_NEGASI:
    stop = stop - NEGASI

if STEMMING and BAHASA == "en":
    from nltk.stem import PorterStemmer
    _stem = PorterStemmer().stem
elif STEMMING and BAHASA == "id":
    try:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        _stem = StemmerFactory().create_stemmer().stem
    except ImportError:
        print("Sastrawi belum terpasang -> stemming dilewati")
        _stem = None
else:
    _stem = None


def bersihkan(teks):
    t = str(teks)
    if LOWERCASE:
        t = t.lower()
    if MASK:                                        # URL, mention, angka -> token generik
        t = re.sub(r"http\S+|www\.\S+|\b\S+\.(?:com|org|net|ly|id|co)\S*", " urltoken ", t)
        t = re.sub(r"@\w+", " usertoken ", t)
        t = re.sub(r"\b\d+\b", " numtoken ", t)
    kata = re.findall(r"[a-zA-Z]+", t)
    if STOPWORD:
        kata = [w for w in kata if w not in stop]
    if _stem:
        kata = [_stem(w) for w in kata]
    return " ".join(kata) if kata else "kosongtoken"


df["clean"] = df["text"].map(bersihkan)
print("sebelum:", df["text"].iloc[0][:70])
print("sesudah:", df["clean"].iloc[0][:70])

sebelum: Ponsel lipat generasi kedua hadir dengan engsel lebih tahan lama
sesudah: ponsel lipat generasi kedua hadir engsel lebih tahan lama


---
## §5 · Split train/test

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=TEST_SIZE,
    stratify=df["label"],          # proporsi kelas terjaga
    random_state=SEED)
print("train:", len(X_train), "| test:", len(X_test))
print("distribusi train:", y_train.value_counts().to_dict())

train: 61 | test: 27
distribusi train: {'olahraga': 16, 'ekonomi': 15, 'politik': 15, 'teknologi': 15}


---
## §6 · Model

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

bobot = "balanced" if SEIMBANGKAN else None
PILIHAN_MODEL = {
    "nb":     MultinomialNB(),
    "logreg": LogisticRegression(max_iter=1000, class_weight=bobot),
    "svm":    LinearSVC(class_weight=bobot),
    "tree":   DecisionTreeClassifier(max_depth=8, class_weight=bobot, random_state=SEED),
    "rf":     RandomForestClassifier(n_estimators=200, class_weight=bobot, random_state=SEED),
}
clf = PILIHAN_MODEL[MODEL]
print("model:", clf.__class__.__name__)

model: LinearSVC


---
## §7 · TF-IDF + latih

`Pipeline` menjamin vectorizer hanya di-`fit` pada data latih — kalau tidak, nilai IDF ikut
"melihat" data uji dan skornya jadi palsu.

In [8]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=NGRAM, min_df=MIN_DF, sublinear_tf=True)),
    ("clf",   clf),
])
model.fit(X_train, y_train)
print("jumlah fitur:", len(model.named_steps["tfidf"].vocabulary_))

jumlah fitur: 812


---
## §8 · Evaluasi

In [9]:
pred = model.predict(X_test)
kelas = sorted(df["label"].unique())

print("akurasi :", round(accuracy_score(y_test, pred), 3))
print("f1 macro:", round(f1_score(y_test, pred, average="macro"), 3), "\n")
print(classification_report(y_test, pred, zero_division=0))

print("Confusion matrix -- baris = aktual, kolom = prediksi:")
print(pd.DataFrame(confusion_matrix(y_test, pred, labels=kelas),
                   index=["true_" + k for k in kelas],
                   columns=["pred_" + k for k in kelas]).to_string())

# kelas mana yang paling sering tertukar
cm = confusion_matrix(y_test, pred, labels=kelas)
np.fill_diagonal(cm, 0)
if cm.sum():
    i, j = np.unravel_index(cm.argmax(), cm.shape)
    print(f"\npaling sering tertukar: '{kelas[i]}' diprediksi '{kelas[j]}' ({cm[i, j]}x)")

# kata khas tiap kelas
nama = np.array(model.named_steps["tfidf"].get_feature_names_out())
c = model.named_steps["clf"]
if hasattr(c, "coef_"):
    print()
    for i, k in enumerate(c.classes_):
        print(f"[{k:10s}]", ", ".join(nama[np.argsort(c.coef_[i])[-8:][::-1]]))

akurasi : 0.852
f1 macro: 0.857 

              precision    recall  f1-score   support

     ekonomi       1.00      0.57      0.73         7
    olahraga       1.00      1.00      1.00         6
     politik       0.64      1.00      0.78         7
   teknologi       1.00      0.86      0.92         7

    accuracy                           0.85        27
   macro avg       0.91      0.86      0.86        27
weighted avg       0.91      0.85      0.85        27

Confusion matrix -- baris = aktual, kolom = prediksi:
                pred_ekonomi  pred_olahraga  pred_politik  pred_teknologi
true_ekonomi               4              0             3               0
true_olahraga              0              6             0               0
true_politik               0              0             7               0
true_teknologi             0              0             1               6

paling sering tertukar: 'ekonomi' diprediksi 'politik' (3x)

[ekonomi   ] sektor, bank, asing, kuartal, pr

---
## §9 · Prediksi teks baru

In [10]:
teks_baru = ["harga saham ditutup menguat pada perdagangan hari ini",
             "gelandang itu mencetak dua gol di babak kedua",
             "aplikasi ini menambahkan enkripsi untuk melindungi data pengguna"]

for t, p in zip(teks_baru, model.predict([bersihkan(t) for t in teks_baru])):
    print(f"  {p:10s} <- {t[:60]}")

  ekonomi    <- harga saham ditutup menguat pada perdagangan hari ini
  olahraga   <- gelandang itu mencetak dua gol di babak kedua
  teknologi  <- aplikasi ini menambahkan enkripsi untuk melindungi data peng


---
## §10 · Output — simpan model & tulis hasil ke file

Tiga keluaran yang biasanya diminta: **model tersimpan**, **file hasil prediksi**, dan
**ringkasan skor**. Model disimpan bersama setelan preprocessing-nya — tanpa itu, model yang
dimuat ulang tidak tahu teks baru harus dibersihkan bagaimana.

In [11]:
import joblib, json

NAMA = PATH.split("/")[-1].split(".")[0]          # nama dasar dari file data

# ---- 1. simpan model + setelan preprocessing ----
joblib.dump({"pipeline": model,
             "prep": dict(BAHASA=BAHASA, LOWERCASE=LOWERCASE, MASK=MASK,
                          STOPWORD=STOPWORD, JAGA_NEGASI=JAGA_NEGASI, STEMMING=STEMMING)},
            f"model_{NAMA}.joblib")

# ---- 2. tulis hasil prediksi data uji ----
hasil = pd.DataFrame({"text": X_test.values, "aktual": y_test.values, "prediksi": pred})
hasil["benar"] = hasil["aktual"] == hasil["prediksi"]
hasil.to_csv(f"hasil_prediksi_{NAMA}.csv", index=False)

# ---- 3. ringkasan skor + setelan, untuk ditempel di laporan ----
ringkas = {
    "dataset": PATH, "model": MODEL,
    "fitur": f"TF-IDF ngram={NGRAM} min_df={MIN_DF}",
    "preprocessing": [k for k, v in [("lowercase", LOWERCASE), ("mask", MASK),
                                     ("stopword", STOPWORD), ("jaga_negasi", JAGA_NEGASI),
                                     ("stemming", STEMMING)] if v],
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "akurasi": round(float(accuracy_score(y_test, pred)), 4),
    "f1_macro": round(float(f1_score(y_test, pred, average="macro")), 4),
}
with open(f"ringkasan_{NAMA}.json", "w") as f:
    json.dump(ringkas, f, indent=2)

print("tersimpan:")
print(f"  model_{NAMA}.joblib          <- model siap dipakai lagi")
print(f"  hasil_prediksi_{NAMA}.csv    <- {len(hasil)} baris: text/aktual/prediksi/benar")
print(f"  ringkasan_{NAMA}.json        <- skor + setelan")
print()
print(json.dumps(ringkas, indent=2))

# ---- cek: model yang dimuat ulang memberi hasil yang sama ----
b = joblib.load(f"model_{NAMA}.joblib")
print("\nmuat ulang cocok:", bool((b["pipeline"].predict(X_test) == pred).all()))

tersimpan:
  model_berita_topik.joblib          <- model siap dipakai lagi
  hasil_prediksi_berita_topik.csv    <- 27 baris: text/aktual/prediksi/benar
  ringkasan_berita_topik.json        <- skor + setelan

{
  "dataset": "data/berita_topik.tsv",
  "model": "svm",
  "fitur": "TF-IDF ngram=(1, 2) min_df=1",
  "preprocessing": [
    "lowercase",
    "mask",
    "stopword",
    "jaga_negasi"
  ],
  "n_train": 61,
  "n_test": 27,
  "akurasi": 0.8519,
  "f1_macro": 0.857
}

muat ulang cocok: True


---
## Catatan khusus case multi-class

- **Kodenya sama persis dengan biner.** sklearn otomatis memakai strategi one-vs-rest; tidak ada
  parameter khusus yang perlu diubah.
- **Yang berubah cuma pembacaan metrik**: pakai `average="macro"` (tiap kelas berbobot sama).
  `average="micro"` sama dengan accuracy pada kasus single-label, jadi tidak menambah informasi.
- **Confusion matrix jadi k×k.** Bacanya per baris: dari semua dokumen kelas X, ke mana saja
  model salah menaruhnya. Pasangan kelas yang sering tertukar biasanya memang mirip topiknya —
  itu temuan yang layak ditulis di laporan.
- Kalau satu dokumen boleh punya lebih dari satu label, pindah ke `cheatsheet-multilabel.ipynb`.